# Exploratory Data Analysis

**Period:** September 4, 2025 → February 25, 2026  
**Source:** Cleaned dataset from `01_data_cleaning.ipynb`

## Objectives
1. Load cleaned data
2. Compute descriptive statistics
3. Identify best/worst days
4. Analyze patterns by day of week
5. Compare weekend vs weekday recovery
6. Explore basic correlations

**Key Questions:**
- What are my average HRV, RHR, Sleep duration?
- Which day had the best/worst recovery?
- Do I recover better on weekends?
- What factors correlate most with Readiness Score?

In [9]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

print("Libraries imported successfully!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Libraries imported successfully!
Pandas: 3.0.1
NumPy: 2.4.2


In [10]:
# LOAD CLEAN DATA

df = pd.read_csv('/Users/tim/Projects/custom-readiness-score/data/oura_clean.csv', index_col='date', parse_dates=True)
print(f"Data loaded: {df.shape[0]} days and {df.shape[1]} columns")

# Period
first_date = df.index.min().strftime('%y/%m/%d')
last_date = df.index.max().strftime('%y/%m/%d')
print(f"Period: {first_date} --> {last_date}")

print("First 5 rows:")
display(df.head())

Data loaded: 164 days and 22 columns
Period: 25/09/04 --> 26/02/25
First 5 rows:


,Unnamed: 0,hrv,light_sleep,total_sleep,sleep_efficiency,sleep_score,temperature,readiness_score,deep_sleep,rem_sleep,rhr,total_sleep_min,deep_sleep_min,rem_sleep_min,light_sleep_min,total_sleep_h,hrv_rhr_ratio,deep_pct,rem_pct,day_of_week,day_name,weekend
date,,,,,,,,,,,,,,,,,,,,,,
2025-09-04,0,66.0,12840.0,22110.0,68.0,52,0.03,73,5850.0,3420.0,55.34,368.5,97.5,57.0,214.0,6.14,1.19,26.46,15.47,3,Thursday,False
2025-09-05,1,67.0,4350.0,10800.0,92.0,53,1.26,57,3990.0,2460.0,52.11,180.0,66.5,41.0,72.5,3.00,1.29,36.94,22.78,4,Friday,False
2025-09-06,2,58.0,10800.0,20490.0,80.0,97,0.37,84,5550.0,4140.0,61.27,341.5,92.5,69.0,180.0,5.69,0.95,27.09,20.20,5,Saturday,True
2025-09-07,3,74.0,10320.0,21420.0,77.0,82,0.35,82,6300.0,4800.0,53.92,357.0,105.0,80.0,172.0,5.95,1.37,29.41,22.41,6,Sunday,True
2025-09-08,4,39.0,10890.0,20130.0,91.0,62,0.62,37,5310.0,3930.0,71.62,335.5,88.5,65.5,181.5,5.59,0.54,26.38,19.52,0,Monday,False


In [11]:
# DESCRIPTIVE STATISTICS

key_metrics = df[['hrv', 'rhr', 'temperature', 'total_sleep_h', 'deep_pct', 'rem_pct', 'sleep_efficiency', 'sleep_score', 'readiness_score', 'hrv_rhr_ratio']]

stats = key_metrics.describe().T.round(2)
display(stats)

print("\nKey Insights:")
print(f"Average HRV: {df['hrv'].mean():.1f} ms, Min HRV: {df['hrv'].min():.0f} ms, Max HRV: {df['hrv'].max():.0f} ")
print(f"Average RHR: {df['rhr'].mean():.1f}, Min RHR: {df['rhr'].min():.0f}, Max RHR: {df['rhr'].max():.0f} ")
print(f"Average Sleep Duration: {df['total_sleep_h'].mean():.1f} hours")
print(f"Average Readiness Score: {df['readiness_score'].mean()}")

,count,mean,std,min,25%,50%,75%,max
hrv,164.0,58.68,8.18,16.00,55.00,59.00,63.00,77.00
rhr,164.0,56.95,5.18,49.76,53.96,56.08,58.44,83.97
temperature,164.0,0.15,0.40,-1.19,-0.10,0.14,0.36,1.86
total_sleep_h,164.0,7.21,1.94,2.12,5.81,7.39,8.59,11.93
deep_pct,164.0,26.91,5.63,7.45,23.79,26.34,30.22,45.10
rem_pct,164.0,23.86,4.71,6.04,21.32,24.17,27.20,34.13
sleep_efficiency,164.0,83.67,9.85,49.00,78.00,86.00,91.00,97.00
sleep_score,164.0,73.35,11.53,41.00,67.00,75.00,82.00,97.00
readiness_score,164.0,74.50,11.39,24.00,69.75,77.00,81.00,93.00
hrv_rhr_ratio,164.0,1.05,0.19,0.19,0.98,1.06,1.16,1.46



Key Insights:
Average HRV: 58.7 ms, Min HRV: 16 ms, Max HRV: 77 
Average RHR: 57.0, Min RHR: 50, Max RHR: 84 
Average Sleep Duration: 7.2 hours
Average Readiness Score: 74.5


In [12]:
# IDENTIFY WORST DAY

print("Worst Day Analysis:\n")

worst_readiness = df['readiness_score'].min()
worst_day = df['readiness_score'].idxmin()

print(f"Worst day: {worst_day.strftime('%Y/%m/%d (%A)')}")
print(f"Readiness Score: {worst_readiness}/100")

worst_day_data = df.loc[worst_day]

print("\nMetrics on worst day vs Average:")

# Key metrics to compare
metrics_to_show = ['hrv', 'rhr', 'temperature', 'total_sleep_h', 'deep_pct', 'rem_pct', 'sleep_score', 'hrv_rhr_ratio']

for metric in metrics_to_show:
    value = worst_day_data[metric]
    avg = df[metric].mean()
    diff_pct = ((value - avg) / avg) * 100
    print(f"{metric:20s}: {value:6.1f} (avg: {avg:6.1f}, {diff_pct:+6.1f}%)")

Worst Day Analysis:

Worst day: 2025/10/19 (Sunday)
Readiness Score: 24/100

Metrics on worst day vs Average:
hrv                 :   43.0 (avg:   58.7,  -26.7%)
rhr                 :   76.8 (avg:   57.0,  +34.8%)
temperature         :    1.5 (avg:    0.1, +913.4%)
total_sleep_h       :    6.2 (avg:    7.2,  -13.4%)
deep_pct            :   16.2 (avg:   26.9,  -40.0%)
rem_pct             :   20.3 (avg:   23.9,  -15.0%)
sleep_score         :   57.0 (avg:   73.3,  -22.3%)
hrv_rhr_ratio       :    0.6 (avg:    1.0,  -46.5%)


In [13]:
# DENTIFY BEST DAY

print("Best Day Analysis:\n")

best_readiness = df['readiness_score'].max()
best_day = df['readiness_score'].idxmax()

print(f"Best Day: {best_day.strftime('%Y/%m/%d (%A)')}")
print(f"Readiness Score: {best_readiness}/100")

best_day_data = df.loc[best_day]

print("\nMetrics on best day vs Average:")

# Key metrics to compare
metrics_to_show = ['hrv', 'rhr', 'temperature', 'total_sleep_h', 'deep_pct', 'rem_pct', 'sleep_score', 'hrv_rhr_ratio']

for metric in metrics_to_show:
    value = best_day_data[metric]
    avg = df[metric].mean()
    diff_pct = ((value - avg) / avg) * 100
    print(f"{metric:20s}: {value:6.1f} (avg: {avg:6.1f}, {diff_pct:+6.1f}%)")

Best Day Analysis:

Best Day: 2026/01/19 (Monday)
Readiness Score: 93/100

Metrics on best day vs Average:
hrv                 :   52.0 (avg:   58.7,  -11.4%)
rhr                 :   54.0 (avg:   57.0,   -5.1%)
temperature         :    0.1 (avg:    0.1,  -38.0%)
total_sleep_h       :   10.7 (avg:    7.2,  +47.9%)
deep_pct            :   23.5 (avg:   26.9,  -12.5%)
rem_pct             :   28.0 (avg:   23.9,  +17.3%)
sleep_score         :   86.0 (avg:   73.3,  +17.2%)
hrv_rhr_ratio       :    1.0 (avg:    1.0,   -8.0%)


In [14]:
# ATTERNS BY DAY OF WEEK

# Group by day of the week
daily_stats = df.groupby('day_of_week')[['hrv', 'rhr', 'total_sleep_h', 'sleep_score', 'readiness_score']].mean()

# Map day numbers to names
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_stats.index = daily_stats.index.map(lambda x: day_names[x])

daily_stats = daily_stats.round(1)

print("Average metrics by days:")
display(daily_stats)

# Find best and worst day for readiness
worst_day_readiness = daily_stats['readiness_score'].idxmin()
best_day_readiness = daily_stats['readiness_score'].idxmax()

print(f"Worst day for recovery: {worst_day_readiness} (Readiness: {daily_stats.loc[worst_day_readiness, 'readiness_score']})")
print(f"Best day for recovery: {best_day_readiness} (Readiness: {daily_stats.loc[best_day_readiness, 'readiness_score']})")

Average metrics by days:


,hrv,rhr,total_sleep_h,sleep_score,readiness_score
day_of_week,,,,,
Monday,59.2,56.4,6.7,68.6,73.0
Tuesday,58.3,57.0,6.5,67.8,71.7
Wednesday,57.4,57.3,6.7,74.8,74.2
Thursday,56.1,58.9,8.0,75.7,74.2
Friday,60.8,56.4,6.3,69.2,72.9
Saturday,58.8,56.1,7.7,78.6,78.0
Sunday,60.3,56.4,8.5,78.7,77.7


Worst day for recovery: Tuesday (Readiness: 71.7)
Best day for recovery: Saturday (Readiness: 78.0)


In [29]:
# WEEKEND VS WEEKDAY COMPARISON

weekend_stats = df.groupby('weekend')[['hrv', 'rhr', 'total_sleep_h', 'sleep_score', 'readiness_score']].mean()
weekend_stats.index = ['Weekday', 'Weekend']
weekend_stats = weekend_stats.round(1)

display(weekend_stats)

for col in weekend_stats.columns:
    weekday_val = weekend_stats.loc['Weekday', col]
    weekend_val = weekend_stats.loc['Weekend', col]

    diff = weekend_val - weekday_val
    diff_pct = (diff / weekday_val) * 100

    if col == 'rhr':
        emoji = "🟢" if diff < 0 else "🔴"  # Lower is better
    else:
        emoji = "🟢" if diff > 0 else "🔴"  # Higher is better
    
    print(f"{emoji} {col:20s}: {diff:+6.1f} ({diff_pct:+5.1f}%)")


,hrv,rhr,total_sleep_h,sleep_score,readiness_score
Weekday,58.3,57.2,6.8,71.3,73.2
Weekend,59.6,56.3,8.2,78.7,77.8


🟢 hrv                 :   +1.3 ( +2.2%)
🟢 rhr                 :   -0.9 ( -1.6%)
🟢 total_sleep_h       :   +1.4 (+20.6%)
🟢 sleep_score         :   +7.4 (+10.4%)
🟢 readiness_score     :   +4.6 ( +6.3%)
